# Movie Recommendation System
## Content-based filtering with Python, TF-IDF and Flask
This notebook loads the two supplied TMDB CSV files, audits their quality, combines content features, builds a cosine-similarity model, evaluates basic behaviour and saves the two files used by Flask.

**Run order:** extract the full project, install `requirements-notebook.txt`, open this notebook from the project folder, and choose **Run → Run All Cells**. The supplied notebook contains captured outputs from sequential Python execution; a Jupyter kernel was unavailable in the build environment.

The credits upload is incomplete and partly damaged. We keep every valid movie using a left join on IDs; missing or malformed credits contribute no cast/director features. The notebook does not invent missing values.

Only load pickle files from trusted sources. Model files are already included, and running this notebook rebuilds them.

### 1. Imports and project location
Install once in a terminal: `python -m pip install -r requirements-notebook.txt`.
TF-IDF provides tokenization, lowercasing and English stop-word removal, so no NLTK corpus download is needed.

In [1]:
from pathlib import Path
from collections import Counter
import json
import pickle
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import sys
print('Python:', sys.version.split()[0])
ROOT = Path.cwd()
if not (ROOT / 'data' / 'tmdb_5000_movies.csv').exists():
    if (ROOT / 'movie-recommender' / 'data').is_dir():
        ROOT = ROOT / 'movie-recommender'
    else:
        raise FileNotFoundError('Open the notebook from the extracted movie-recommender folder.')
print('Data files located.')

Python: 3.12.13
Data files located.


### 2. Load, validate and explore
Read identifiers as text before validation. Read only the four named credit columns because this upload contains many empty spreadsheet columns. Invalid credit IDs are excluded; duplicate IDs cause a clear error instead of multiplying rows. Movie titles are not unique identifiers.

In [2]:
def load_data(data_dir):
    """Retain all valid movies; never merge by potentially ambiguous title."""
    data_dir = Path(data_dir)
    movies = pd.read_csv(data_dir / 'tmdb_5000_movies.csv', dtype=str, keep_default_na=False)
    credits = pd.read_csv(data_dir / 'tmdb_5000_credits.csv', dtype=str,
                          keep_default_na=False, usecols=['movie_id', 'title', 'cast', 'crew'])
    required = {'id', 'title', 'genres', 'keywords', 'overview', 'release_date'}
    if required - set(movies):
        raise ValueError(f'Missing movie columns: {sorted(required - set(movies))}')
    report = {'movie_rows_raw': len(movies), 'credit_rows_raw': len(credits)}
    for frame, key in [(movies, 'id'), (credits, 'movie_id')]:
        ids = pd.to_numeric(frame[key], errors='coerce')
        valid = ids.notna() & (ids % 1 == 0) & (ids > 0)
        frame.drop(index=frame.index[~valid], inplace=True)
        frame[key] = ids.loc[valid].astype('int64')
    movies = movies.loc[movies.title.str.strip().ne('')].copy()
    if movies.id.duplicated().any() or credits.movie_id.duplicated().any():
        raise ValueError('Duplicate movie IDs found; resolve these before merging.')
    report['valid_credit_rows'] = len(credits)
    report['discarded_credit_rows'] = report['credit_rows_raw'] - len(credits)
    merged = movies.merge(credits[['movie_id', 'cast', 'crew']], how='left',
                          left_on='id', right_on='movie_id', validate='one_to_one', indicator=True)
    report['movies_without_credit_row'] = int(merged['_merge'].eq('left_only').sum())
    merged = merged.drop(columns=['movie_id', '_merge']).fillna('').reset_index(drop=True)
    report['movies_retained'] = len(merged)
    if len(merged) < 2:
        raise ValueError('At least two movies are required.')
    return merged, report

raw_movies, quality = load_data(ROOT / 'data')
print(json.dumps(quality, indent=2))
print(raw_movies[['id', 'title', 'release_date']].head().to_string(index=False))
print('Missing overviews:', int(raw_movies.overview.eq('').sum()))
print('Duplicate titles:', int(raw_movies.title.duplicated().sum()))
print('Most common original languages:')
print(raw_movies.original_language.value_counts().head().to_string())

{
  "movie_rows_raw": 4803,
  "credit_rows_raw": 3863,
  "valid_credit_rows": 1492,
  "discarded_credit_rows": 2371,
  "movies_without_credit_row": 3313,
  "movies_retained": 4803
}
    id                                    title release_date
 19995                                   Avatar   2009-12-10
   285 Pirates of the Caribbean: At World's End   2007-05-19
206647                                  Spectre   2015-10-26
 49026                    The Dark Knight Rises   2012-07-16
 49529                              John Carter   2012-03-07
Missing overviews: 3
Duplicate titles: 3
Most common original languages:
original_language
en    4505
fr      70
es      32
zh      27
de      27


### 3. Clean JSON fields and create tags
Tags combine the overview, genres, keywords, the first three cast members and directors. Structured phrases are kept together (`Science Fiction` becomes `sciencefiction`) so they act as one feature. JSON errors are counted and treated as missing metadata. We use JSON parsing, never `eval`.

In [3]:
def build_tags(movies, report):
    """Malformed JSON becomes an empty list and is counted, not guessed."""
    movies = movies.copy()
    errors = Counter()
    def parse(value, field):
        if not value:
            return []
        try:
            result = json.loads(value)
            if not isinstance(result, list) or any(not isinstance(x, dict) for x in result):
                raise ValueError('Expected a list of objects')
            return result
        except (ValueError, TypeError):
            errors[field] += 1
            return []
    def names(items):
        return [x['name'].strip() for x in items if isinstance(x.get('name'), str) and x['name'].strip()]
    def atom(value):
        return re.sub(r'\W+', '', value.casefold())
    for field in ['genres', 'keywords', 'cast', 'crew']:
        movies[field + '_parsed'] = [parse(v, field) for v in movies[field]]
    movies['genre_names'] = movies.genres_parsed.map(names)
    movies['cast_names'] = movies.cast_parsed.map(lambda x: names(x[:3]))
    movies['director_names'] = movies.crew_parsed.map(lambda x: names([p for p in x if p.get('job') == 'Director']))
    def tags(row):
        structured = row['genre_names'] + names(row['keywords_parsed']) + row['cast_names'] + row['director_names']
        # Keep multiword entities together: science fiction -> sciencefiction.
        return ' '.join([str(row['overview']).casefold()] + [atom(x) for x in structured if atom(x)])
    movies['tags'] = movies.apply(tags, axis=1)
    movies['year'] = movies.release_date.str.extract(r'^(\d{4})', expand=False).fillna('Unknown year')
    report['malformed_json_by_field'] = {f: errors[f] for f in ['genres', 'keywords', 'cast', 'crew']}
    report['movies_without_usable_cast'] = int(movies.cast_names.map(len).eq(0).sum())
    report['movies_without_usable_director'] = int(movies.director_names.map(len).eq(0).sum())
    report['empty_tags'] = int(movies.tags.str.strip().eq('').sum())
    keep = ['id', 'title', 'year', 'overview', 'genre_names', 'cast_names', 'director_names', 'tags']
    return movies[keep].reset_index(drop=True)

movies = build_tags(raw_movies, quality)
print(json.dumps(quality, indent=2))
print(movies[['title','genre_names','cast_names']].head().to_string(index=False))
print('Example tags:', movies.iloc[0].tags[:650])

{
  "movie_rows_raw": 4803,
  "credit_rows_raw": 3863,
  "valid_credit_rows": 1492,
  "discarded_credit_rows": 2371,
  "movies_without_credit_row": 3313,
  "movies_retained": 4803,
  "malformed_json_by_field": {
    "genres": 0,
    "keywords": 0,
    "cast": 3,
    "crew": 6
  },
  "movies_without_usable_cast": 3316,
  "movies_without_usable_director": 3322,
  "empty_tags": 0
}
                                   title                                   genre_names                                       cast_names
                                  Avatar [Action, Adventure, Fantasy, Science Fiction] [Sam Worthington, Zoe Saldana, Sigourney Weaver]
Pirates of the Caribbean: At World's End                  [Adventure, Fantasy, Action]    [Johnny Depp, Orlando Bloom, Keira Knightley]
                                 Spectre                    [Action, Adventure, Crime]     [Daniel Craig, Christoph Waltz, Léa Seydoux]
                   The Dark Knight Rises              [Action, Crime, Dram

### 4. TF-IDF and cosine similarity
TF-IDF downweights terms that appear in many movies. Cosine similarity compares the direction of two feature vectors: a higher value means more shared content. The vocabulary is capped at 10,000 features, and float32 reduces the saved matrix size.

This is unsupervised content retrieval, not prediction of a numerical rating. There is no label-based training/test split in this baseline. Fitting on the available catalogue is appropriate for recommending items from that same catalogue.

In [4]:
def train_model(movies):
    vectorizer = TfidfVectorizer(stop_words='english', max_features=10000,
                                 sublinear_tf=True, dtype=np.float32)
    vectors = vectorizer.fit_transform(movies.tags)
    similarity = cosine_similarity(vectors).astype(np.float32)
    np.clip(similarity, 0, 1, out=similarity)
    return vectorizer, vectors, similarity

vectorizer, vectors, similarity = train_model(movies)
quality['vocabulary_size'] = vectors.shape[1]
quality['similarity_shape'] = list(similarity.shape)
print('Feature matrix:', vectors.shape)
print('Cosine similarity matrix:', similarity.shape)
print('Matrix memory: %.1f MiB' % (similarity.nbytes / 1024**2))
print('Movies with no vocabulary features:', int(np.asarray(vectors.getnnz(axis=1) == 0).sum()))

Feature matrix: (4803, 10000)
Cosine similarity matrix: (4803, 4803)
Matrix memory: 88.0 MiB
Movies with no vocabulary features: 0


### 5. Recommend by movie ID or title
We explicitly remove the selected movie's ID, even when multiple movies have tied similarity scores. The title helper ignores case and surrounding spaces. If a title belongs to multiple movies, choose a unique movie ID; the Flask dropdown includes year and ID.

In [5]:
def recommend_by_id(movie_id, movies, similarity, n=5):
    if not isinstance(n, int) or not 1 <= n <= 20:
        raise ValueError('Choose between 1 and 20 recommendations.')
    matches = np.flatnonzero(movies.id.to_numpy() == int(movie_id))
    if len(matches) != 1:
        raise ValueError('Please choose a movie from the list.')
    index = int(matches[0])
    scores = similarity[index]
    order = np.argsort(-scores, kind='stable')
    order = [int(i) for i in order if i != index and scores[i] > 0][:n]
    result = movies.iloc[order].copy()
    result['similarity'] = scores[order]
    return result

def recommend(title, movies, similarity, n=5):
    matches = movies.loc[movies.title.str.casefold().eq(str(title).strip().casefold())]
    if matches.empty:
        raise ValueError(f'Movie not found: {title}')
    if len(matches) > 1:
        raise ValueError('Multiple films share this title; use recommend_by_id with a movie ID.')
    return recommend_by_id(int(matches.iloc[0].id), movies, similarity, n)

for title in ['Avatar', 'The Dark Knight', 'Toy Story']:
    print('\nSimilar to', title)
    print(recommend(title, movies, similarity)[['id','title','similarity']].to_string(index=False))


Similar to Avatar
    id                   title  similarity
270938           Falcon Rising    0.139597
 54138 Star Trek Into Darkness    0.138751
228326        The Book of Life    0.134402
  8077                  Alien³    0.123490
  7450              Titan A.E.    0.120036

Similar to The Dark Knight
    id                                   title  similarity
 49026                   The Dark Knight Rises    0.364412
   272                           Batman Begins    0.306411
   364                          Batman Returns    0.264266
   414                          Batman Forever    0.261395
142061 Batman: The Dark Knight Returns, Part 2    0.255793

Similar to Toy Story
   id                  title  similarity
10193            Toy Story 3    0.409975
  863            Toy Story 2    0.382258
 6957 The 40 Year Old Virgin    0.180601
14799 For Your Consideration    0.137288
11551         Small Soldiers    0.128291


### 6. Evaluation and limitations
Check the matrix, ordering, duplicate prevention and self-exclusion. Then estimate catalogue coverage and genre overlap on 200 deterministically sampled movies.

**These are diagnostics, not recommendation accuracy.** Genre overlap is optimistic because genres are model inputs. Coverage describes diversity of recommended items; it does not show user satisfaction. Proper precision@k, recall@k or NDCG needs held-out user relevance judgements or interaction data, which these files do not contain. Read the example recommendations and judge their relevance rather than treating cosine scores as probabilities.

In [6]:
assert similarity.shape == (len(movies), len(movies))
assert np.isfinite(similarity).all()
assert np.allclose(similarity, similarity.T, atol=1e-6)
assert float(similarity.min()) >= 0 and float(similarity.max()) <= 1
rng = np.random.default_rng(42)
sample_ids = rng.choice(movies.id.to_numpy(), size=min(200, len(movies)), replace=False)
recommended_ids, overlaps = set(), []
for movie_id in sample_ids:
    result = recommend_by_id(int(movie_id), movies, similarity)
    assert movie_id not in result.id.to_numpy()
    assert result.id.is_unique and result.similarity.is_monotonic_decreasing
    recommended_ids.update(result.id.tolist())
    base_genres = set(movies.loc[movies.id.eq(movie_id), 'genre_names'].iloc[0])
    for candidate_genres in result.genre_names:
        if base_genres and candidate_genres:
            overlaps.append(bool(base_genres.intersection(candidate_genres)))
print('Matrix and recommendation checks passed for', len(sample_ids), 'sampled movies.')
print('Sampled top-5 catalogue coverage: %.2f%%' % (100 * len(recommended_ids) / len(movies)))
print('Eligible recommendation pairs with a shared genre: %.2f%%' % (100 * np.mean(overlaps)))
print('Interpret these figures as descriptive diagnostics only.')

Matrix and recommendation checks passed for 200 sampled movies.
Sampled top-5 catalogue coverage: 18.11%
Eligible recommendation pairs with a shared genre: 81.77%
Interpret these figures as descriptive diagnostics only.


### 7. Save model artifacts
`model.pkl` stores cleaned movie metadata in the same order as the rows and columns of `similarity.pkl`. Rebuild both together after changing data or preprocessing. `train.py` provides the same pipeline outside Jupyter.

In [7]:
def save_model(movies, similarity, directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    # Only load pickle files produced by this project or another trusted source.
    for name, value in [('model.pkl', movies), ('similarity.pkl', similarity)]:
        with (directory / name).open('wb') as handle:
            pickle.dump(value, handle, protocol=4)

def load_model(directory):
    directory = Path(directory)
    try:
        with (directory / 'model.pkl').open('rb') as handle:
            movies = pickle.load(handle)
        with (directory / 'similarity.pkl').open('rb') as handle:
            similarity = pickle.load(handle)
    except FileNotFoundError as exc:
        raise RuntimeError('Model files are missing. Run python train.py or all notebook cells first.') from exc
    if not isinstance(movies, pd.DataFrame) or not isinstance(similarity, np.ndarray):
        raise ValueError('Invalid model files. Rebuild with python train.py.')
    if similarity.shape != (len(movies), len(movies)) or movies.id.duplicated().any():
        raise ValueError('Model files do not match. Rebuild with python train.py.')
    return movies.reset_index(drop=True), similarity

save_model(movies, similarity, ROOT)
(ROOT / 'data_quality_report.json').write_text(json.dumps(quality, indent=2), encoding='utf-8')
loaded_movies, loaded_similarity = load_model(ROOT)
assert loaded_movies.id.tolist() == movies.id.tolist()
assert np.array_equal(loaded_similarity, similarity)
print('Saved and verified model.pkl and similarity.pkl.')

Saved and verified model.pkl and similarity.pkl.


### 8. Run the Flask application
In a separate terminal, from this folder:
```bash
python app.py
```
Open http://localhost:5000. `/` is the homepage and `/recommend` accepts the movie selection. This address works on your computer; it is not a public deployment.

For Docker, see `README.md`. The Dockerfile rebuilds the artifacts in the container and serves Flask using Gunicorn. Docker was unavailable in the build environment, so the container still needs a local build test.

### Bonus extension plan: collaborative and hybrid recommenders
These optional models are **not implemented** in this submission baseline because the uploads contain no user IDs or individual user ratings. `vote_average` is a movie-level aggregate, not a user–movie rating matrix.

To extend the project, use real ratings with `userId`, `movieId`, `rating` and preferably a timestamp. For MovieLens, map its movie IDs to TMDB IDs using its links file before combining scores.

1. Split interactions into train/test per user before computing similarities.
2. Compute user means from observed training ratings. Build a mean-centred user–movie matrix, keeping a separate observed-rating mask.
3. Compute cosine or Pearson similarity between users; require co-rated items and shrink similarities for low overlap.
4. Predict each unseen movie from neighbours who actually rated it, adding the target user's mean back to the weighted centred score. Use a training-only popularity fallback for cold starts.
5. Build a content profile from the user's positively rated movies. Normalize content and collaborative scores onto compatible scales, then blend, for example `0.6 * collaborative + 0.4 * content`.
6. Tune the weight on validation interactions and report ranking metrics on held-out interactions. Never count already-seen movies as recommendations.

SVD is matrix factorization, not user-based nearest-neighbour filtering. If the requirement is specifically user-based filtering, use a user-neighbour implementation such as user-based KNN.

### References and data provenance
- Original movies attachment: `39a983f6-e03d-424d-bdd5-735ebcb430b6.csv`, copied unchanged to `data/tmdb_5000_movies.csv`.
- Original credits attachment: `862dddd0-0111-4563-ba5b-fe36620696f7.csv`, copied unchanged to `data/tmdb_5000_credits.csv`.
- [Scikit-learn TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- [Scikit-learn cosine similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)
- [Flask deployment guidance](https://flask.palletsprojects.com/en/stable/deploying/)
- [Flask with Gunicorn](https://flask.palletsprojects.com/en/stable/deploying/gunicorn/)

Main limitations: uneven credit coverage, missing metadata, bag-of-words loss of context, English stop words on a multilingual catalogue, and no personal preference data. The full similarity matrix scales quadratically; a larger catalogue should use on-demand similarity or nearest-neighbour search.